# Smoke test: value-guided KV cache engine

First end-to-end run: load DeepSeek-R1-Distill-Qwen-7B, run a handful of GSM8K
examples through the custom decode loop under each baseline policy plus the
entropy-salience policy, and sanity-check accuracy/throughput/memory numbers
before scaling up to a full eval sweep.

Run this on the remote GPU box (L4/L40). Install deps first:
`pip install -r requirements.txt && pip install -e .`

In [ ]:
import sys
sys.path.insert(0, "../src")

import torch
from vgkv.model import load_model_and_tokenizer, DEFAULT_MODEL_ID
from vgkv.eval.gsm8k import load_gsm8k, build_prompt, is_correct
from vgkv.eval.runner import PolicySpec, run_policy_eval, summarize
from vgkv.decode import generate_with_policy, DecodeConfig
from vgkv.value_models import NoEviction, RandomPolicy, RecencyPolicy, H2OPolicy, EntropySaliencePolicy

print(torch.cuda.get_device_name(0))

In [2]:
!pip install -r requirements.txt && pip install -e .
!pip uninstall -y torchvision torchaudio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 150.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 305.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 526.6/526.6 MB 294.6 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 366.2/366.2 MB 285.1 MB/s  0:00:010:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.1/170.1 MB 313.3 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.0/206.0 MB 322.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.4/60.4 MB 297.9 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 197.6/197.6 MB 308.9 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 341.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 MB 365.8 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 355.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 MB 337.9 MB/s  0

In [1]:
import os
os.environ["HF_HOME"] = "/home/jovyan/shared/jd.tan/.cache/huggingface"

import sys
sys.path.insert(0, "../src")

import torch
from vgkv.model import load_model_and_tokenizer, DEFAULT_MODEL_ID
from vgkv.eval.gsm8k import load_gsm8k, build_prompt, is_correct
from vgkv.decode import generate_with_policy, DecodeConfig
from vgkv.value_models import NoEviction, RandomPolicy, RecencyPolicy, H2OPolicy, EntropySaliencePolicy

print(torch.cuda.get_device_name(0))

NVIDIA L40S


In [2]:
model, tokenizer = load_model_and_tokenizer(DEFAULT_MODEL_ID, device="cuda")

2026-08-16 05:40:29.405522: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-16 05:40:29.431176: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-16 05:40:29.431205: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-16 05:40:29.432161: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-16 05:40:29.436686: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructio

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [4]:
ds = load_gsm8k(split="test", limit=5)
example = ds[0]
prompt = build_prompt(example)
print(prompt)

Solve the following grade-school math problem. Think step by step, then give the final numeric answer on its own line after '#### '.

Problem: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?



## Single-example sanity check across policies

Use a generous budget first (>= expected total sequence length) with `NoEviction`
to confirm the decode loop + eager attention plumbing works and matches
`model.generate()` output for the same prompt (greedy decoding should be
identical). Then drop the budget and compare policies.

In [ ]:
# R1-Distill produces long <think> traces -- 256 tokens isn't enough to reach
# a "#### <answer>" line on most GSM8K problems, so bump max_new_tokens way up
# for this reference run.
MAX_NEW_TOKENS = 1536

config_full = DecodeConfig(max_new_tokens=MAX_NEW_TOKENS, budget=10_000, sink_size=4)
text, metrics = generate_with_policy(model, tokenizer, prompt, NoEviction(), config_full)
print(text)
print(metrics.as_dict())
print("correct:", is_correct(text, example))

In [ ]:
# cross-check against generate() for identical greedy output on the same prompt
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    ref_out = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
ref_text = tokenizer.decode(ref_out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(ref_text)
print("matches custom loop:", ref_text == text)

## Tight-budget comparison across policies

Pick a generation budget well below `MAX_NEW_TOKENS` so eviction actually kicks
in, then compare accuracy/throughput/memory across policies on a few examples.

The full prompt is always protected (`protect_prompt=True`, the default) --
eviction only ever competes over which *generated reasoning* tokens to keep,
not whether the model can still recall the original word problem. Budgets are
expressed as `generation_budget` (how many generated tokens may survive
eviction) rather than an absolute cache size, since GSM8K prompt lengths vary
per example and an absolute budget would silently give longer-prompt examples
less room to reason than shorter-prompt ones.

`no_eviction` gets `generation_budget=MAX_NEW_TOKENS` so eviction never
triggers for it (a genuine full-cache reference) -- `NoEviction.score()`
returns a uniform score for every token, so if it were given the same tight
budget as the other policies, eviction would still trigger once the cache
exceeds budget and ties would be broken by arbitrary `torch.topk` order,
silently turning "no eviction" into a near-random policy.

Each policy is constructed fresh per example via a factory function, not a
shared instance -- `EntropySaliencePolicy` accumulates per-sequence attention
state on the policy object itself (not inside `ManagedKVCache`), so reusing
one instance across examples would leak attention mass from one problem into
the next example's scores.

In [ ]:
# Room for reasoning tokens under eviction -- the full prompt is always
# protected on top of this (protect_prompt=True), so this is purely how many
# *generated* tokens a policy gets to keep resident at once. Started at 128,
# bumped after the first sweep showed 128 was too tight for any policy to
# separate from 0% -- see PROJECT.md approach log for the diagnosis.
TIGHT_GENERATION_BUDGET = 384

policy_specs = [
    PolicySpec("no_eviction", lambda: NoEviction(), generation_budget=MAX_NEW_TOKENS),
    PolicySpec("random", lambda: RandomPolicy(seed=0), generation_budget=TIGHT_GENERATION_BUDGET),
    PolicySpec("recency_lru", lambda: RecencyPolicy(), generation_budget=TIGHT_GENERATION_BUDGET),
    PolicySpec("h2o", lambda: H2OPolicy(), generation_budget=TIGHT_GENERATION_BUDGET),
    PolicySpec("entropy_salience", lambda: EntropySaliencePolicy(), generation_budget=TIGHT_GENERATION_BUDGET),
]

small_ds = load_gsm8k(split="test", limit=5)
results = run_policy_eval(model, tokenizer, small_ds, policy_specs, max_new_tokens=MAX_NEW_TOKENS)
results

In [ ]:
summarize(results)

## Scaled-up eval: larger subset, generation-budget sweep

Now run the same comparison across more GSM8K examples and a range of
generation budgets, to start building the accuracy-vs-budget curve per policy.
Start with `EVAL_N` examples and `GENERATION_BUDGETS` below -- both are
deliberately small the first time you run this cell on a fresh box, since
total runtime scales as
`EVAL_N * len(GENERATION_BUDGETS) * len(policies) * MAX_NEW_TOKENS`. Increase
once you've confirmed timing is reasonable (check `avg_prefill_s`/`avg_tok_s`
from the first budget's summary before letting the rest run).

`no_eviction` is only run once per example (at `generation_budget=MAX_NEW_TOKENS`,
i.e. eviction never triggers), not once per budget in the sweep -- it doesn't
depend on budget by construction, so re-running it at every sweep point would
just waste compute re-deriving the same ceiling number.

In [ ]:
EVAL_N = 30  # bump to 100-1319 (full test set) once timing looks reasonable
GENERATION_BUDGETS = [64, 128, 256, 512]

eval_ds = load_gsm8k(split="test", limit=EVAL_N)

evictable_policy_factories = {
    "random": lambda: RandomPolicy(seed=0),
    "recency_lru": lambda: RecencyPolicy(),
    "h2o": lambda: H2OPolicy(),
    "entropy_salience": lambda: EntropySaliencePolicy(),
}

# no_eviction: single reference run per example, independent of the budget sweep
no_eviction_specs = [PolicySpec("no_eviction", lambda: NoEviction(), generation_budget=MAX_NEW_TOKENS)]
no_eviction_results = run_policy_eval(
    model, tokenizer, eval_ds, no_eviction_specs, max_new_tokens=MAX_NEW_TOKENS, verbose=False
)

# evictable policies: one run per (example, generation_budget, policy)
sweep_results = [no_eviction_results]
for gen_budget in GENERATION_BUDGETS:
    specs = [
        PolicySpec(name, factory, generation_budget=gen_budget)
        for name, factory in evictable_policy_factories.items()
    ]
    df = run_policy_eval(model, tokenizer, eval_ds, specs, max_new_tokens=MAX_NEW_TOKENS, verbose=False)
    sweep_results.append(df)
    print(f"generation_budget={gen_budget} done")

import pandas as pd
all_results = pd.concat(sweep_results, ignore_index=True)
all_results.to_csv("../results/gsm8k_policy_sweep.csv", index=False)
summarize(all_results)

In [ ]:
import matplotlib.pyplot as plt

summary = summarize(all_results)
no_evict_acc = summary.loc[summary["policy"] == "no_eviction", "accuracy"].iloc[0]

fig, ax = plt.subplots(figsize=(7, 5))
for policy_name in evictable_policy_factories:
    sub = summary[summary["policy"] == policy_name].sort_values("generation_budget")
    ax.plot(sub["generation_budget"], sub["accuracy"], marker="o", label=policy_name)

ax.axhline(no_evict_acc, color="black", linestyle="--", label="no_eviction (full cache)")
ax.set_xlabel("Generation budget (tokens kept resident, prompt always protected)")
ax.set_ylabel("GSM8K accuracy")
ax.set_title(f"Accuracy vs. generation budget (n={EVAL_N})")
ax.legend()
plt.savefig("../results/accuracy_vs_budget.png", dpi=150, bbox_inches="tight")
plt.show()

## Next steps

- Increase `EVAL_N` toward the full 1319-example test set once per-example
  timing (`avg_prefill_s` / `avg_tok_s` in the summary) confirms the full sweep
  is tractable in your session time budget
- Widen `GENERATION_BUDGETS` (e.g. add 32, 1024) to fill out the
  accuracy-vs-budget curve
- Results are logged to `../results/gsm8k_policy_sweep.csv` (raw, per-example)
  and `../results/accuracy_vs_budget.png` (plot) each run -- diff successive
  CSVs when iterating on `EntropySaliencePolicy` to see if a change actually
  moved the curve
- Update `../PROJECT.md` section 6 approach log with findings before trying the
  next value-model idea (attention-graph centrality, PRM-based step scoring)